# Voice Denoising via SVD on Hankel Matrices — Pipeline v1

End-to-end pipeline:
1. Load audio
2. Build Hankel matrix
3. Apply truncated SVD
4. Reconstruct signal via anti-diagonal averaging
5. Save denoised output

In [ ]:
import numpy as np
import scipy.linalg as la
import librosa
import soundfile as sf
from pathlib import Path

DATA_DIR = Path("data")
INPUT_PATH  = DATA_DIR / "recording-1.wav"
OUTPUT_PATH = DATA_DIR / "recording-1-denoised.wav"

# --- Configuration ---
L = 2048   # Hankel window length (rows); K = N - L + 1 columns
K_RANK = 10  # number of singular values to keep

## 1. Load audio

In [4]:
# Load as mono float32, preserving original sample rate
signal, sr = librosa.load(INPUT_PATH, sr=None, mono=True)

print(f"Sample rate : {sr} Hz")
print(f"Duration    : {len(signal)/sr:.2f} s  ({len(signal)} samples)")
print(f"dtype       : {signal.dtype}")
print(f"Amplitude   : [{signal.min():.4f}, {signal.max():.4f}]")

Sample rate : 48000 Hz
Duration    : 2.08 s  (99671 samples)
dtype       : float32
Amplitude   : [-0.3893, 0.4738]


## 2. Hankel matrix composition

Given a signal $x = (x_1, \dots, x_N)$ and window length $L$, the Hankel matrix is

$$
H_{ij} = x_{i+j-1}, \quad i = 1,\dots,L,\; j = 1,\dots,K,\; K = N - L + 1.
$$

Each anti-diagonal holds the same sample index, which is the key property exploited during reconstruction.

In [5]:
def build_hankel(x: np.ndarray, L: int) -> np.ndarray:
    """Construct the L x (N-L+1) Hankel embedding matrix."""
    N = len(x)
    K = N - L + 1
    if K < 1:
        raise ValueError(f"Window length L={L} exceeds signal length N={N}.")
    # Use stride tricks for zero-copy construction
    shape   = (L, K)
    strides = (x.strides[0], x.strides[0])
    return np.lib.stride_tricks.as_strided(x, shape=shape, strides=strides)

H = build_hankel(signal, L)
print(f"Hankel matrix shape : {H.shape}  (L={L}, K={H.shape[1]})")
print(f"Memory (view)       : {H.nbytes / 1e6:.1f} MB")

Hankel matrix shape : (2048, 97624)  (L=2048, K=97624)
Memory (view)       : 799.7 MB


## 3. SVD algorithm application

Compute the full SVD $H = U \Sigma V^T$ and retain only the top-$k$ components:

$$
H_k = \sum_{i=1}^{k} \sigma_i \, u_i v_i^T.
$$

In [6]:
# scipy.linalg.svd returns full matrices by default; use full_matrices=False
# (economy / thin SVD) to keep shapes manageable.
# We make a writeable copy of H first because as_strided returns a read-only view.
H_copy = np.array(H, dtype=np.float64)

U, s, Vt = la.svd(H_copy, full_matrices=False)

print(f"U  shape : {U.shape}")
print(f"s  shape : {s.shape}  (total singular values)")
print(f"Vt shape : {Vt.shape}")
print(f"\nTop-10 singular values : {s[:10].round(2)}")
print(f"Energy captured by top-{K_RANK}: {(s[:K_RANK]**2).sum() / (s**2).sum() * 100:.1f} %")

U  shape : (2048, 2048)
s  shape : (2048,)  (total singular values)
Vt shape : (2048, 97624)

Top-10 singular values : [184.32 184.   183.96 174.02 159.94 159.29 129.43 129.38 127.93 127.3 ]
Energy captured by top-50: 80.8 %


## 4. Reconstruction

Form the rank-$k$ approximation $H_k$, then recover the 1-D signal by **anti-diagonal averaging** (Hankelization): each sample index $n$ corresponds to the anti-diagonal $\{(i,j) : i+j = n+1\}$; the reconstructed value is the mean of all entries on that anti-diagonal.

In [7]:
def low_rank_approx(U: np.ndarray, s: np.ndarray, Vt: np.ndarray, k: int) -> np.ndarray:
    """Reconstruct the rank-k approximation U_k @ diag(s_k) @ Vt_k."""
    return (U[:, :k] * s[:k]) @ Vt[:k, :]


def hankel_to_signal(Hk: np.ndarray) -> np.ndarray:
    """Anti-diagonal averaging: recover 1-D signal from a (possibly non-Hankel) matrix."""
    L, K = Hk.shape
    N = L + K - 1
    out = np.zeros(N)
    counts = np.zeros(N)
    for i in range(L):
        out[i : i + K]    += Hk[i, :]
        counts[i : i + K] += 1
    return out / counts


Hk = low_rank_approx(U, s, Vt, K_RANK)
denoised = hankel_to_signal(Hk)

# Clip to [-1, 1] to avoid clipping artifacts when saving as float
denoised = np.clip(denoised, -1.0, 1.0).astype(np.float32)

print(f"Reconstructed signal length : {len(denoised)} samples  (original: {len(signal)})")
print(f"Amplitude range             : [{denoised.min():.4f}, {denoised.max():.4f}]")

Reconstructed signal length : 99671 samples  (original: 99671)
Amplitude range             : [-0.3889, 0.3580]


## 5. Save denoised audio

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sf.write(OUTPUT_PATH, denoised, sr)
print(f"Saved denoised audio → {OUTPUT_PATH}")
print(f"Duration : {len(denoised)/sr:.2f} s at {sr} Hz")

Saved denoised audio → data/recording-1-denoised.wav
Duration : 2.08 s at 48000 Hz
